In [4]:
import numpy as np
import pandas as pd

In [5]:
df = pd.read_csv("fordgobike-tripdata.csv")

print("Original shape:", df.shape)
df.info()

Original shape: (183416, 16)
<class 'pandas.DataFrame'>
RangeIndex: 183416 entries, 0 to 183415
Data columns (total 16 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   duration_sec             183416 non-null  int64  
 1   start_time               183416 non-null  str    
 2   end_time                 183416 non-null  str    
 3   start_station_id         183219 non-null  float64
 4   start_station_name       183219 non-null  str    
 5   start_station_latitude   183416 non-null  float64
 6   start_station_longitude  183416 non-null  float64
 7   end_station_id           183219 non-null  float64
 8   end_station_name         183219 non-null  str    
 9   end_station_latitude     183416 non-null  float64
 10  end_station_longitude    183416 non-null  float64
 11  bike_id                  183416 non-null  int64  
 12  user_type                183416 non-null  str    
 13  member_birth_year        175151 non-null 

In [6]:
# Remove duplicate rows
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

Shape after removing duplicates: (183412, 16)
Remaining duplicates: 0


In [11]:
# Convert ID columns to string
df["start_station_id"] = df["start_station_id"].astype("string")
df["end_station_id"] = df["end_station_id"].astype("string")
df["bike_id"] = df["bike_id"].astype("string")

# Convert time columns to datetime
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["end_time"] = pd.to_datetime(df["end_time"], errors="coerce")

print(df[["start_time", "end_time"]].head())

           start_time            end_time
0                 NaT 2026-09-10 01:56:00
1                 NaT                 NaT
2 2026-09-10 13:13:12                 NaT
3                 NaT 2026-09-10 02:36:48
4                 NaT 2026-09-10 20:44:06


In [13]:
missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values[missing_values > 0])

Missing values:
start_time            109501
end_time              110151
start_station_id         197
start_station_name       197
end_station_id           197
end_station_name         197
member_birth_year       8265
member_gender           8265
start_time_td         183412
end_time_td           183412
start_minute          183412
end_minute            183412
dtype: int64


In [14]:
df = df.dropna(
    subset=[
        "start_station_id",
        "start_station_name",
        "end_station_id",
        "end_station_name"
    ]
)

print("Shape after removing missing station information:", df.shape)

Shape after removing missing station information: (183215, 20)


In [17]:
df["member_gender"] = df["member_gender"].fillna("Unknown")

In [18]:
data_year = 2019

df["member_age"] = data_year - df["member_birth_year"]

# Replace unrealistic ages with NaN
df.loc[
    (df["member_age"] < 10) | (df["member_age"] > 85),
    "member_age"
] = np.nan

print(df["member_age"].describe())

count    174765.000000
mean         34.126610
std           9.881006
min          18.000000
25%          27.000000
50%          32.000000
75%          39.000000
max          85.000000
Name: member_age, dtype: float64


In [19]:
# Convert trip duration from seconds to minutes
df["duration_min"] = df["duration_sec"] / 60

print(df["duration_min"].describe())

count    183215.000000
mean         12.098367
std          29.917978
min           1.016667
25%           5.416667
50%           8.566667
75%          13.266667
max        1424.066667
Name: duration_min, dtype: float64


In [20]:
# Remove invalid trip durations
df = df[
    (df["duration_sec"] > 0) &
    (df["duration_sec"] <= 3600)
]

print("Shape after duration filtering:", df.shape)

Shape after duration filtering: (181509, 22)


In [21]:
# Extract weekday
df["weekday"] = df["start_time"].dt.day_name()

# Weekend flag
df["is_weekend"] = df["start_time"].dt.dayofweek >= 5

print(df[["start_time", "weekday", "is_weekend"]].head())

  start_time weekday  is_weekend
4        NaT     NaN       False
5        NaT     NaN       False
6        NaT     NaN       False
7        NaT     NaN       False
8        NaT     NaN       False


In [22]:
bins = [0, 18, 30, 45, 60, 100]
labels = ["Under 18", "18-30", "31-45", "46-60", "60+"]

df["age_group"] = pd.cut(
    df["member_age"],
    bins=bins,
    labels=labels
)

print(df["age_group"].value_counts(dropna=False))

age_group
31-45       76353
18-30       73644
46-60       20036
NaN          8130
60+          3317
Under 18       29
Name: count, dtype: int64


In [23]:
# Standardize categorical columns
df["user_type"] = df["user_type"].astype("category")
df["member_gender"] = df["member_gender"].astype("category")

print(df["user_type"].value_counts())
print(df["member_gender"].value_counts())

user_type
Subscriber    162715
Customer       18794
Name: count, dtype: int64
member_gender
Male       129574
Female      40419
Unknown      7943
Other        3573
Name: count, dtype: int64


In [24]:
# Encode user type
df["user_type_encoded"] = df["user_type"].cat.codes

print(df[["user_type", "user_type_encoded"]].drop_duplicates())

     user_type  user_type_encoded
4   Subscriber                  1
12    Customer                  0


In [26]:
print("Final shape:", df.shape)

print("\nRemaining Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nData Types:")
print(df.dtypes)

Final shape: (181509, 26)

Remaining Missing Values:
start_time           108386
end_time             108998
member_birth_year      7943
start_time_td        181509
end_time_td          181509
start_minute         181509
end_minute           181509
member_age             8130
weekday              108386
age_group              8130
dtype: int64

Data Types:
duration_sec                        int64
start_time                 datetime64[us]
end_time                   datetime64[us]
start_station_id                   string
start_station_name                    str
start_station_latitude            float64
start_station_longitude           float64
end_station_id                     string
end_station_name                      str
end_station_latitude              float64
end_station_longitude             float64
bike_id                            string
user_type                        category
member_birth_year                 float64
member_gender                    category
bike_share_

In [27]:
df.to_csv("cleaned_fordgobike_data.csv", index=False)

print("Successfully saved to cleaned_fordgobike_data.csv!")

Successfully saved to cleaned_fordgobike_data.csv!
